# 05 — MCP Tool-Calling Agent

**Purpose:** Prove the whole pipeline end-to-end — an LLM discovers the 3 tools over the
real MCP protocol (not a hardcoded function map) and decides which one to call for each
natural-language request: answer a question via RAG, write an analysis report, or classify
a new applicant.

### What This Notebook Does
1. Connects a `DatabricksMCPClient` to the managed MCP endpoint from
   `src/04_agent_tools_mcp_functions` and lists the discovered tools.
2. Converts the MCP tool schemas into OpenAI-style tool definitions for a Foundation
   Model API chat model.
3. Runs 3 demo prompts through a tool-calling loop — one per tool — and prints the full
   transcript (tool chosen, arguments, tool result, final answer).

### Source & Target
| Direction | Resource |
|-----------|----------|
| Source | Managed MCP endpoint `/api/2.0/mcp/functions/insurance_rag_agent/agent_tools` |
| Target | Printed tool-call transcript (no table write — this is the agent demo) |

> **Prerequisites:** Run `src/04_agent_tools_mcp_functions` first.

---

In [ ]:
%pip install -U databricks-mcp mcp openai
dbutils.library.restartPython()

In [ ]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
CATALOG              = 'insurance_rag_agent'
AGENT_TOOLS_SCHEMA    = 'agent_tools'
# Get CHAT_MODEL_ENDPOINT and MODEL_QUERY_BASE_PATH from Serving > (your chat model) >
# "Use this model" > the model= and base_url= values in the generated code snippet — see
# docs/SETUP.md step 7 for the two naming patterns we've seen (AI Gateway vs. classic).
CHAT_MODEL_ENDPOINT    = f'{CATALOG}.agent_tools.meta-llama-3-3-70b-instruct_backend'
MODEL_QUERY_BASE_PATH  = '/ai-gateway/mlflow/v1'

DEMO_PROMPTS = [
    'What do the records show about smokers in the southeast region with high charges?',
    'Generate an analysis report comparing smokers vs. non-smokers in the southeast.',
    'Classify this applicant: 52-year-old smoker, BMI 34, from the southeast, with 2 children.',
]

print(f'Chat model endpoint: {CHAT_MODEL_ENDPOINT}')
print(f'Demo prompts: {len(DEMO_PROMPTS)}')

In [ ]:
import json

from databricks.sdk import WorkspaceClient
from databricks_mcp import DatabricksMCPClient
from openai import OpenAI

w = WorkspaceClient()

# This workspace proxies Foundation Model calls through AI Gateway rather than the
# classic serving_endpoints.query() path, so tool-calling goes through the OpenAI client.
openai_client = OpenAI(
    api_key  = w.config.token,
    base_url = f'{w.config.host}{MODEL_QUERY_BASE_PATH}',
)

---

In [0]:
mcp_server_url = f'{w.config.host}/api/2.0/mcp/functions/{CATALOG}/{AGENT_TOOLS_SCHEMA}'

mcp_client = DatabricksMCPClient(server_url=mcp_server_url, workspace_client=w)

discovered_tools = mcp_client.list_tools()

for tool in discovered_tools:
    print(f'- {tool.name}: {tool.description}')

---

In [0]:
openai_tools = [
    {
        'type': 'function',
        'function': {
            'name':        tool.name,
            'description': tool.description,
            'parameters':  tool.inputSchema,
        },
    }
    for tool in discovered_tools
]

print(json.dumps(openai_tools, indent=2))

---

In [ ]:
def run_agent_turn(user_message):
    '''One tool-calling round trip: ask the model, run any MCP tool it picks, ask again for the final answer.'''
    messages = [{'role': 'user', 'content': user_message}]
    print(f'\n=== User: {user_message} ===')

    first_response = openai_client.chat.completions.create(
        model    = CHAT_MODEL_ENDPOINT,
        messages = messages,
        tools    = openai_tools,
    )
    choice = first_response.choices[0].message

    if not choice.tool_calls:
        print(f'Assistant: {choice.content}')
        return

    messages.append({
        'role':       'assistant',
        'content':    choice.content,
        'tool_calls': [
            {
                'id':       tc.id,
                'type':     'function',
                'function': {'name': tc.function.name, 'arguments': tc.function.arguments},
            }
            for tc in choice.tool_calls
        ],
    })

    for tool_call in choice.tool_calls:
        tool_name = tool_call.function.name
        tool_args = json.loads(tool_call.function.arguments)
        print(f'--> Calling MCP tool: {tool_name}({tool_args})')

        tool_result = mcp_client.call_tool(tool_name, tool_args)
        result_text = tool_result.content[0].text
        print(f'<-- Tool result: {result_text[:300]}')

        messages.append({
            'role':         'tool',
            'tool_call_id': tool_call.id,
            'content':      result_text,
        })

    final_response = openai_client.chat.completions.create(model=CHAT_MODEL_ENDPOINT, messages=messages)
    print(f'Assistant: {final_response.choices[0].message.content}')


for prompt in DEMO_PROMPTS:
    run_agent_turn(prompt)

---

In [0]:
print(f'Demo complete — {len(DEMO_PROMPTS)} prompts routed across {len(discovered_tools)} MCP tools.')